### Overview

This notebook shows how the input to ASAHI is made: one matrix of betas and one of read depth per chromosome, with one row per canonical hg38 CpG and one column per library.

#### The full pipeline, from FASTQ
1. **FASTQ -> strand-merged CpG file:** `assets/bioinfopipe_pairedWGBS.sh` runs Trim Galore, Bismark alignment, deduplication, methylation extraction and strand merging. Single-end commands are in its header. Pass `--keep-intermediates` to keep the deduplicated BAM.
2. **Deduplicated BAM -> low-depth versions:** `assets/downsample_bam.sh` runs `samtools view` at each fraction (0.1, 0.2, 0.4, 0.6), then the same methylation calling.
3. **Strand-merged CpG files -> matrices:** `scripts/build_matrices.py`, which this notebook runs.

Accessions and FASTQ links for the 12 ENCODE libraries are in `data/metadata/encode_libraries.tsv`.

#### What is included
- `data/encode_subset/`: output of step 1 for the 12 libraries at full depth, chr21 only. Used below.
- `data/encode_pt*/`: finished chr21 matrices at full depth and at the 4 low depths. The other notebooks use these.

chr1 and chr11 are not included. To rebuild them, run all 3 steps on the ENCODE FASTQs, then the last cell below for the chromosome you want.

In [1]:
%cd -q ..

In [2]:
import gzip
from asahi.process_dataset import make_cc_gt

### Build the chr21 full-depth matrices
A CpG missing from a library's file is left empty in its column.

In [3]:
fm, cov = make_cc_gt(data_dir="data/encode_subset",
                     chrom="chr21",
                     index_path="data/metadata/hg38_cpg_py0idx.csv.gz",
                     fm_save_path="data/encode_pt_rebuilt/chrom_centric/true/FractionalMethylation/chr21.fm",
                     cov_save_path="data/encode_pt_rebuilt/chrom_centric/true/ReadDepth/chr21.cov")
fm.iloc[100000:100005, :4]

chr21: 12 samples, 462299 canonical CpGs


  0%|          | 0/12 [00:00<?, ?it/s]

  8%|▊         | 1/12 [00:00<00:06,  1.73it/s]

 75%|███████▌  | 9/12 [00:01<00:00,  9.88it/s]

100%|██████████| 12/12 [00:01<00:00, 10.77it/s]

CpGs observed: mean 85.652%, s.d. 2.570%


saved data/encode_pt_rebuilt/chrom_centric/true/FractionalMethylation/chr21.fm


saved data/encode_pt_rebuilt/chrom_centric/true/ReadDepth/chr21.cov


,ENCFF032LIY,ENCFF055UXZ,ENCFF116KBT,ENCFF164ZJM
start,,,,
13578781,0.740741,0.538462,0.857143,0.571429
13578786,0.346154,0.564103,0.642857,0.500000
13578830,0.571429,0.552632,0.538462,0.105263
13578924,0.758621,0.710526,0.800000,0.500000
13578944,0.968750,0.950000,1.000000,1.000000


In [4]:
cov.iloc[100000:100005, :4]

,ENCFF032LIY,ENCFF055UXZ,ENCFF116KBT,ENCFF164ZJM
start,,,,
13578781,27.0,39.0,14.0,21.0
13578786,26.0,39.0,14.0,20.0
13578830,28.0,38.0,13.0,19.0
13578924,29.0,38.0,20.0,14.0
13578944,32.0,40.0,17.0,12.0


### Check against the matrices shipped in `data/encode_pt`

In [5]:
def same_file(rebuilt, shipped):
    with open(rebuilt, "rb") as a, gzip.open(shipped, "rb") as b:
        return a.read() == b.read()

new, old = "data/encode_pt_rebuilt/chrom_centric/true", "data/encode_pt/chrom_centric/true"
print("shape:", fm.shape)
print("betas byte-identical:", same_file(f"{new}/FractionalMethylation/chr21.fm", f"{old}/FractionalMethylation/chr21.fm.gz"))
print("depths byte-identical:", same_file(f"{new}/ReadDepth/chr21.cov", f"{old}/ReadDepth/chr21.cov.gz"))

shape: (462299, 12)


betas byte-identical: True
depths byte-identical: True


### Other chromosomes and the genome
Not run here, since they need the output of steps 1 and 2:
```bash
python scripts/build_matrices.py --cov-dir <dir with one subdirectory per sample> --dataset encode_pt --chrom chr1

# The model's DNA branch needs the encoded chromosome. chr21 is included; for others:
python scripts/encode_genome.py hg38.fa data/metadata/hg38_encoded.npz --chroms chr1 chr11
# (add --prepend-chr, with --chroms 1 11, if the FASTA headers read "1" rather than "chr1")
```